To extract the unique categories from the OSM and Foursquare datasets, you can use the following SQL commands. These commands create new tables containing distinct category information and export them to CSV files.

```sql
CREATE TABLE osm_categories AS
SELECT DISTINCT osm_class, osm_type FROM osm ;
\copy osm_categories TO 'osm_categories.csv' WITH ( FORMAT csv, HEADER, DELIMITER ',', QUOTE '"', ESCAPE '"', NULL '' );
```


Foursquare can be downloaded from the Foursquare website: https://docs.foursquare.com/data-products/docs/categories
(data/fsq_personalization-apis-movement-sdk-categories.csv)

Then we feed data/fsq_personalization-apis-movement-sdk-categories.csv, osm_categories.csv and https://wiki.openstreetmap.org/wiki/Map_features to `GPT 5.5 with High Intelligence` to generate a mapping between Foursquare categories and OSM categories. Resulted in a mapping file fsq_osm_category_mapping.csv. 

prompt: "I want you to add two columns to fsq_personalization-apis-movement-sdk-categories.csv that are osm_class, osm_type that they have the same meaning. If ou cannot find a match from osm please put nf_fsq for both osm_class, osm_type."

In [ ]:
import pandas
fsq_osm_category_mapping = pandas.read_csv('../../data/fsq_osm_category_mapping.csv')
fsq_personalization = pandas.read_csv('../../data/fsq_personalization-apis-movement-sdk-categories.csv')

In [2]:

# check that the two files have the same [Category ID, Category Name, Category Label] values 
f1 = fsq_osm_category_mapping[['Category ID', 'Category Name', 'Category Label']].sort_values(by=['Category ID', 'Category Name', 'Category Label']).reset_index(drop=True)
f2 = fsq_personalization[['Category ID', 'Category Name', 'Category Label']].sort_values(by=['Category ID', 'Category Name', 'Category Label']).reset_index(drop=True)
assert f1.equals(f2), "The two files do not have the same [Category ID, Category Name, Category Label] values. Please check the files for discrepancies."

In [3]:
category_mapping_dict = {}
for index, row in fsq_osm_category_mapping.iterrows():
    category_mapping_dict[row['Category ID']] = {
                                            'Category Name': row['Category Name'],
                                            'Category Label': row['Category Label'],
                                            "osm_class": row['osm_class'],
                                             "osm_type": row['osm_type']}


In [ ]:
world_poi_path='../../data/World_POI_levenshtein_0.3.csv'
world_poi_df = pandas.read_csv(world_poi_path, chunksize=100_000)

In [ ]:
# Precompute faster lookup version once
fast_category_mapping = {
    str(cat_id): {
        "osm_class": set(v.get("osm_class", [])),
        "osm_type": set(v.get("osm_type", []))
    }
    for cat_id, v in category_mapping_dict.items()
}


def parse_cat_ids(value):
    # Handles strings like "[123 456]", "[123, 456]", or "123 456"
    return str(value).strip("[]").replace(",", " ").split()


def get_similarity(fsq_category_ids, osm_class, osm_type):
    cat_ids = parse_cat_ids(fsq_category_ids)

    for cat_id in cat_ids:
        mapping = fast_category_mapping.get(str(cat_id))
        if mapping is None:
            continue

        if osm_class in mapping["osm_class"] or osm_type in mapping["osm_type"]:
            return "yes"

    return "no"


output_path = world_poi_path.replace('.csv', '_category_matched.csv')
first_chunk = True

for chunk in world_poi_df:
    chunk = chunk.dropna(subset=["fsq_category_ids"]).copy()

    chunk["semantic_similarity"] = [
        get_similarity(fsq_ids, osm_class, osm_type)
        for fsq_ids, osm_class, osm_type in zip(
            chunk["fsq_category_ids"],
            chunk["osm_class"],
            chunk["osm_type"]
        )
    ]

    chunk.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

/tmp/ipykernel_501333/3685552415.py:33: DtypeWarning: Columns (0: fsq_post_town) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in world_poi_df:
/tmp/ipykernel_501333/3685552415.py:33: DtypeWarning: Columns (0: fsq_postcode, 1: fsq_post_town) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in world_poi_df:
/tmp/ipykernel_501333/3685552415.py:33: DtypeWarning: Columns (0: fsq_post_town) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in world_poi_df:
/tmp/ipykernel_501333/3685552415.py:33: DtypeWarning: Columns (0: fsq_post_town) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in world_poi_df:
/tmp/ipykernel_501333/3685552415.py:33: DtypeWarning: Columns (0: fsq_post_town) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in world_poi_df:
/tmp/ipykernel_501333/3685552415.py:33: DtypeWarning: Columns (